# **2. Limpieza de datos del dataset de Spotify**

**Objetivo:** Evaluar y tratar los problemas identificados durante la exploración inicial para obtener un conjunto de datos consistente y adecuado para desarrollar un sistema de recomendación musical.

**Fuente de datos:** Dataset de Kaggle: Spotify Tracks Dataset - Audio Features.

**Archivo de entrada:** `spotify-tracks-dataset-detailed.csv`

**Procedimiento:**
1. Cargar los datos originales y verificar su estructura.
2. Revisar los duplicados exactos y determinar si es necesario eliminar las apariciones redundantes.
3. Analizar los identificadores de canciones repetidos y distinguir las diferencias de información entre registros.
4. Investigar los valores numéricos inusuales para determinar si requieren alguna corrección, exclusión o tratamiento específico.
5. Aplicar las transformaciones justificadas y validar el conjunto de datos resultante.
6. Guardar los datos limpios en un archivo independiente del original.

**Criterio de trabajo:** Las decisiones de limpieza se fundamentarán en los resultados de las comprobaciones y en el significado de las variables. No se eliminarán registros ni se modificarán valores únicamente por presentar características inusuales. Se distinguirán los duplicados exactos de los identificadores repetidos que contienen información diferente.

**Reproducibilidad:** El archivo CSV original se conservará sin modificaciones. Las transformaciones se realizarán mediante código en este notebook, registrando los criterios aplicados para que el proceso pueda repetirse y verificarse.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ruta_datos = Path.cwd().parent / 'data' / 'spotify-tracks-dataset-detailed.csv'

print('Ruta del archivo:', ruta_datos)
print('¿Existe el archivo?', ruta_datos.is_file())

Ruta del archivo: c:\Users\VIRIDIANA\Desktop\Proyectos\spotify-music-recommender\data\spotify-tracks-dataset-detailed.csv
¿Existe el archivo? True


In [3]:
df = pd.read_csv(ruta_datos)

print('Tipo de objeto:', type(df))
print('Dimensiones:', df.shape)

Tipo de objeto: <class 'pandas.DataFrame'>
Dimensiones: (114000, 20)


In [4]:
df.isna().sum()

track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [5]:
df.loc[df.isna().any(axis = 1)]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [6]:
filas_iniciales = len(df)
print('Filas antes de la limpieza:', filas_iniciales)

df = df.dropna(subset = ['artists', 'album_name', 'track_name']).copy()

print('\nFilas después de la limpieza:', len(df))
print('Filas eliminadas:', filas_iniciales - len(df))

print('\nValores faltantes en las columnas de identificación:')
print(df[['artists', 'album_name', 'track_name']].isna().sum())

Filas antes de la limpieza: 114000

Filas después de la limpieza: 113999
Filas eliminadas: 1

Valores faltantes en las columnas de identificación:
artists       0
album_name    0
track_name    0
dtype: int64


### **2.1. Tratamiento de valores faltantes**

Durante la exploración inicial se identificó un registro con valores nulos en las columnas `artists`, `album_name` y `track_name`. Debido a que estas variables permiten identificar información fundamental de la canción, se decidió eliminar la fila completa. La operación se aplica únicamente al DataFrame en memoria, sin modificar el archivo CSV original.

In [7]:
duplicados_exactos = df.duplicated()

print('Apariciones duplicadas adicionales:', duplicados_exactos.sum())

Apariciones duplicadas adicionales: 450


In [8]:
grupos_duplicados = df[df.duplicated(keep = False)]

print('Filas pertenecientes a grupos de duplicados:', len(grupos_duplicados))

Filas pertenecientes a grupos de duplicados: 894


In [9]:
frecuencia_duplicados = df.value_counts()

frecuencia_duplicados[frecuencia_duplicados > 1].value_counts().sort_index()

count
2    438
3      6
Name: count, dtype: int64

In [10]:
filas_antes = len(df)
df = df.drop_duplicates().copy()
filas_despues = len(df)

print('Filas antes de eliminar duplicados:', filas_antes)
print('Filas después de eliminar duplicados:', filas_despues)
print('Duplicados eliminados:', filas_antes - filas_despues)
print('¿Quedan duplicados exactos?:', df.duplicated().any())

Filas antes de eliminar duplicados: 113999
Filas después de eliminar duplicados: 113549
Duplicados eliminados: 450
¿Quedan duplicados exactos?: False


### **2.2 Eliminación de duplicados exactos**

Se identificaron 450 apariciones adicionales de registros idénticos en las 20 columnas del conjunto de datos. Estos registros no aportaban información nueva, ya que cada uno tenía al menos una copia con exactamente los mismos valores.

Por este motivo, se decidió conservar una sola aparición de cada registro y eliminar las repeticiones mediante el método `drop_duplicates()` de Pandas. Esta decisión permite evitar que las mismas observaciones tengan un peso innecesario en el análisis posterior y en el desarrollo del sistema de recomendación musical.

Después de la transformación, el conjunto de datos pasó de 113,999 a 113,549 registros. Finalmente, se verificó que no quedaran duplicados exactos mediante `df.duplicated().any()`, cuyo resultado fue `False`.

Esta operación se limitó a los registros idénticos en todas sus columnas. Los casos en los que un mismo `track_id` aparece asociado a valores diferentes se analizarán por separado, ya que podrían contener información relevante para el proyecto.

In [11]:
print('Registros actuales:', len(df))
print('Identificadores únicos:', df['track_id'].nunique())
print('Apariciones adicionales de identificadores:', df['track_id'].duplicated().sum())

Registros actuales: 113549
Identificadores únicos: 89740
Apariciones adicionales de identificadores: 23809


In [12]:
frecuencia_track_id = df['track_id'].value_counts()
ids_repetidos = frecuencia_track_id[frecuencia_track_id > 1]

print('Identificadores que aparecen varias veces:', len(ids_repetidos))
print('Máxima frecuencia de un track_id:', frecuencia_track_id.max())

Identificadores que aparecen varias veces: 16299
Máxima frecuencia de un track_id: 9


In [13]:
columnas_comparacion = []
for columna in df.columns:
    if columna != 'track_id':
        columnas_comparacion.append(columna)

variacion_por_id = (
    df[df['track_id'].duplicated(keep = False)]
    .groupby('track_id')[columnas_comparacion]
    .nunique()
)

print('\nColumnas con diferencias:')
print(variacion_por_id.columns[variacion_por_id.gt(1).any()])


Columnas con diferencias:
Index(['popularity', 'track_genre'], dtype='str')


In [14]:
generos_por_id = (
    df[df['track_id'].duplicated(keep = False)]
    .groupby('track_id')['track_genre']
    .nunique()
)

print('IDs con más de un género:', (generos_por_id > 1).sum())

print('\nDistribución de géneros distintos por ID:')
print(generos_por_id.value_counts().sort_index())

IDs con más de un género: 16299

Distribución de géneros distintos por ID:
track_genre
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64


In [15]:
popularidad_por_id = (
    df[df['track_id'].duplicated(keep = False)]
    .groupby('track_id')['popularity']
    .nunique()
)

print('IDs con más de un valor de popularidad:', (popularidad_por_id > 1).sum())

print('\nDistribución de valores distintos de popularidad:')
print(popularidad_por_id.value_counts().sort_index())

IDs con más de un valor de popularidad: 720

Distribución de valores distintos de popularidad:
popularity
1    15579
2      720
Name: count, dtype: int64


### **2.3 Análisis de identificadores repetidos**

Después de eliminar los duplicados exactos, se identificaron 16,299 valores de `track_id` asociados a más de un registro. Al comparar las columnas, se observó que las únicas variables que presentan diferencias dentro de estos grupos son `track_genre` y `popularity`.

En cuanto al género, los 16,299 identificadores repetidos están asociados a dos o más géneros distintos, con un máximo de nueve géneros por identificador. Esto indica que un mismo identificador puede tener varias asociaciones de género dentro del conjunto de datos.

Por otra parte, 720 identificadores presentan dos valores distintos de popularidad, mientras que los 15,579 restantes mantienen un valor constante.

Se decidió conservar estos registros, ya que las diferencias observadas no demuestran por sí mismas que sean errores. En particular, eliminar las filas asociadas a géneros diferentes podría ocasionar una pérdida de información relevante para el sistema de recomendación. Las posibles diferencias en popularidad se tendrán en cuenta al definir el tratamiento de esta variable en las etapas posteriores.

In [16]:
# Número de valores distintos de popularidad por ID
popularidad_por_id = (
    df.groupby('track_id')['popularity']
    .nunique()
)

# IDs con popularidad constante o variable
ids_popularidad_constante = popularidad_por_id[popularidad_por_id == 1].index

ids_popularidad_variable = popularidad_por_id[popularidad_por_id > 1].index

print('IDs con popularidad constante:', len(ids_popularidad_constante))

print('IDs con popularidad variable:', len(ids_popularidad_variable))

IDs con popularidad constante: 89020
IDs con popularidad variable: 720


In [17]:
# Identificar los IDs que aparecen en varias filas
frecuencia_por_id = df['track_id'].value_counts()

ids_repetidos = frecuencia_por_id[frecuencia_por_id > 1].index

# Seleccionar registros con IDs repetidos y popularidad constante
df_generos = df[df['track_id'].isin(ids_repetidos) & df['track_id'].isin(ids_popularidad_constante)].copy()

print('Registros seleccionados:', len(df_generos))
print('IDs seleccionados:', df_generos['track_id'].nunique())

Registros seleccionados: 38180
IDs seleccionados: 15579


In [18]:
generos_consolidados = (
    df_generos.groupby('track_id')['track_genre']
    .agg(lambda generos: ';'.join(sorted(generos.unique())))
    .reset_index()
)

print('Filas antes de consolidar:', len(df_generos))
print('Filas después de consolidar:', len(generos_consolidados))
print('IDs únicos después de consolidar:', generos_consolidados['track_id'].nunique())

Filas antes de consolidar: 38180
Filas después de consolidar: 15579
IDs únicos después de consolidar: 15579


### **2.3.1 Consolidación de géneros para IDs con popularidad constante**

Se identificaron 15,579 identificadores repetidos que presentan varios valores de `track_genre`, pero mantienen un único valor de `popularity`. Estos identificadores estaban distribuidos en 38,180 filas.

Para evitar que una misma canción aparezca en múltiples registros únicamente por estar asociada a distintos géneros, se agruparon las filas por `track_id` y se reunieron sus géneros distintos en una sola cadena de texto, separados por punto y coma. Los géneros se ordenaron alfabéticamente para obtener resultados consistentes.

Como la popularidad era constante dentro de cada grupo, no fue necesario modificar esta variable. La transformación redujo los registros seleccionados de 38,180 a 15,579, conservando las asociaciones de género identificadas.

Esta consolidación se realizó sobre un DataFrame auxiliar, sin modificar todavía el conjunto de datos principal. Los identificadores con valores distintos de popularidad se tratarán por separado antes de integrar los resultados.

In [19]:
popularidad_por_id = df.groupby('track_id')['popularity'].nunique()

print(popularidad_por_id.value_counts().sort_index())

popularity
1    89020
2      720
Name: count, dtype: int64


In [20]:
# Identificar los IDs con popularidad variable
ids_popularidad_variable = popularidad_por_id[popularidad_por_id > 1].index

# Obtener los pares únicos de ID y popularidad
df_popularidad_variable = (
    df.loc[
        df['track_id'].isin(ids_popularidad_variable),
        ['track_id', 'popularity']
    ]
    .drop_duplicates()
)

# Calcular el promedio por ID y redondearlo
popularidad_consolidada = (
    df_popularidad_variable
    .groupby('track_id')['popularity']
    .mean()
    .add(0.5)
    .apply(np.floor)
    .astype('int64')
    .reset_index()
)

print('Filas consolidadas:', len(popularidad_consolidada))
print('IDs únicos:', popularidad_consolidada['track_id'].nunique())
print('Tipo de popularity:', popularidad_consolidada['popularity'].dtype)
print('Valores faltantes:', popularidad_consolidada['popularity'].isna().sum())

Filas consolidadas: 720
IDs únicos: 720
Tipo de popularity: int64
Valores faltantes: 0


In [21]:
# Seleccionar los registros de los IDs con popularidad variable
df_generos_variables = df[
    df['track_id'].isin(ids_popularidad_variable)
].copy()

# Consolidar los géneros por ID
generos_variables_consolidados = (
    df_generos_variables
    .groupby('track_id')['track_genre']
    .agg(lambda generos: ';'.join(sorted(generos.unique())))
    .reset_index()
)

print('Filas antes de consolidar:', len(df_generos_variables))
print('Filas después de consolidar:', len(generos_variables_consolidados))
print('IDs únicos:', generos_variables_consolidados['track_id'].nunique())

Filas antes de consolidar: 1928
Filas después de consolidar: 720
IDs únicos: 720


### **2.3.2 Consolidación de géneros y popularidad para IDs con valores variables**

Se identificaron 720 identificadores repetidos que presentan varios valores de `track_genre` y dos valores distintos de `popularity`. Estos identificadores estaban distribuidos en 1,928 filas.

Para consolidar los registros, se agruparon las filas por `track_id` y se reunieron los géneros distintos en una sola cadena de texto, separados por punto y coma y ordenados alfabéticamente. De esta manera, se conservaron las asociaciones de género identificadas para cada canción.

A diferencia del grupo anterior, estos identificadores presentaban variaciones en la popularidad. Por ello, se calculó el promedio de los valores distintos de `popularity` asociados a cada identificador y se redondeó al entero más cercano. Para evitar que un valor tuviera mayor peso por aparecer en varias filas, el promedio se calculó después de eliminar las combinaciones repetidas de `track_id` y `popularity`.

La consolidación redujo los registros seleccionados de 1,928 a 720, obteniendo una sola fila por identificador. Tanto los géneros como la popularidad representativa se prepararon en DataFrames auxiliares, sin modificar todavía el conjunto de datos principal.

In [22]:
# Identificar los IDs que aparecen una sola vez
ids_unicos = frecuencia_por_id[frecuencia_por_id == 1].index

print('Cantidad de IDs que aparecen una sola vez:', len(ids_unicos))

Cantidad de IDs que aparecen una sola vez: 73441


In [23]:
df_ids_unicos = df[df['track_id'].isin(ids_unicos)].copy()

print('Filas:', len(df_ids_unicos))
print('IDs únicos:', df_ids_unicos['track_id'].nunique())
print('Columnas:', len(df_ids_unicos.columns))

Filas: 73441
IDs únicos: 73441
Columnas: 20


In [24]:
# Identificar los IDs que aparecen más de una vez
ids_repetidos = frecuencia_por_id[frecuencia_por_id > 1].index

print('Cantidad de IDs repetidos:', len(ids_repetidos))

Cantidad de IDs repetidos: 16299


In [25]:
# Seleccionar los registros de los IDs repetidos
df_ids_repetidos = df[df['track_id'].isin(ids_repetidos)].copy()

print('Filas:', len(df_ids_repetidos))
print('IDs únicos:', df_ids_repetidos['track_id'].nunique())
print('Columnas:', len(df_ids_repetidos.columns))

Filas: 40108
IDs únicos: 16299
Columnas: 20


In [26]:
# Crear una fila de referencia por cada ID repetido
df_repetidos_base = (
    df_ids_repetidos
    .drop_duplicates(subset = 'track_id')
    .copy()
)

print('Filas:', len(df_repetidos_base))
print('IDs únicos:', df_repetidos_base['track_id'].nunique())
print('Columnas:', len(df_repetidos_base.columns))

Filas: 16299
IDs únicos: 16299
Columnas: 20


In [27]:
# Unificar los géneros consolidados de ambos grupos
generos_finales = pd.concat(
    [generos_consolidados, generos_variables_consolidados],
    ignore_index = True
)

print('Filas:', len(generos_finales))
print('IDs únicos:', generos_finales['track_id'].nunique())
print('Columnas:', len(generos_finales.columns))
print('IDs duplicados:', generos_finales['track_id'].duplicated().sum())

Filas: 16299
IDs únicos: 16299
Columnas: 2
IDs duplicados: 0


In [28]:
df_repetidos_base = (
    df_repetidos_base
    .drop(columns = 'track_genre')
    .merge(
        generos_finales,
        on = 'track_id',
        how = 'left',
        validate = 'one_to_one'
    )
)

print('Filas:', len(df_repetidos_base))
print('IDs únicos:', df_repetidos_base['track_id'].nunique())
print('Columnas:', len(df_repetidos_base.columns))
print('IDs duplicados:', df_repetidos_base['track_id'].duplicated().sum())

Filas: 16299
IDs únicos: 16299
Columnas: 20
IDs duplicados: 0


In [29]:
popularidad_constante = (
    df_ids_repetidos[~df_ids_repetidos['track_id'].isin(ids_popularidad_variable)]
    [['track_id', 'popularity']]
    .drop_duplicates(subset = 'track_id')
    .copy()
)

print('Filas:', len(popularidad_constante))
print('IDs únicos:', popularidad_constante['track_id'].nunique())

Filas: 15579
IDs únicos: 15579


In [30]:
popularidad_finales = pd.concat(
    [popularidad_constante, popularidad_consolidada],
    ignore_index = True
)

print('Filas:', len(popularidad_finales))
print('IDs únicos:', popularidad_finales['track_id'].nunique())
print('Columnas:', len(popularidad_finales.columns))
print('Popularidades faltantes:', popularidad_finales['popularity'].isna().sum())
print('IDs duplicados:', popularidad_finales['track_id'].duplicated().sum())
print('Tipo de dato:', popularidad_finales['popularity'].dtype)

Filas: 16299
IDs únicos: 16299
Columnas: 2
Popularidades faltantes: 0
IDs duplicados: 0
Tipo de dato: int64


In [31]:
df_repetidos_base = (
    df_repetidos_base
    .drop(columns = 'popularity')
    .merge(
        popularidad_finales,
        on = 'track_id',
        how  = 'left',
        validate = 'one_to_one'
    )
)

print('Filas:', len(df_repetidos_base))
print('IDs únicos:', df_repetidos_base['track_id'].nunique())
print('Columnas:', len(df_repetidos_base.columns))
print('Popularidades faltantes:', df_repetidos_base['popularity'].isna().sum())
print('IDs duplicados:', df_repetidos_base['track_id'].duplicated().sum())

Filas: 16299
IDs únicos: 16299
Columnas: 20
Popularidades faltantes: 0
IDs duplicados: 0


In [32]:
df_catalogo = pd.concat([df_ids_unicos, df_repetidos_base], ignore_index = True)

print('Filas:', len(df_catalogo))
print('IDs únicos:', df_catalogo['track_id'].nunique())
print('Columnas:', len(df_catalogo.columns))
print('IDs duplicados:', df_catalogo['track_id'].duplicated().sum())
print('Géneros faltantes:', df_catalogo['track_genre'].isna().sum())
print('Popularidades faltantes:', df_catalogo['popularity'].isna().sum())

Filas: 89740
IDs únicos: 89740
Columnas: 20
IDs duplicados: 0
Géneros faltantes: 0
Popularidades faltantes: 0


### **2.4 Comprobación de calidad de la limpieza**

In [33]:
df_catalogo.info()

<class 'pandas.DataFrame'>
RangeIndex: 89740 entries, 0 to 89739
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          89740 non-null  str    
 1   artists           89740 non-null  str    
 2   album_name        89740 non-null  str    
 3   track_name        89740 non-null  str    
 4   popularity        89740 non-null  int64  
 5   duration_ms       89740 non-null  int64  
 6   explicit          89740 non-null  bool   
 7   danceability      89740 non-null  float64
 8   energy            89740 non-null  float64
 9   key               89740 non-null  int64  
 10  loudness          89740 non-null  float64
 11  mode              89740 non-null  int64  
 12  speechiness       89740 non-null  float64
 13  acousticness      89740 non-null  float64
 14  instrumentalness  89740 non-null  float64
 15  liveness          89740 non-null  float64
 16  valence           89740 non-null  float64
 17  temp

In [34]:
df_catalogo.describe().T

,count,mean,std,min,25%,50%,75%,max
popularity,89740.0,33.204558,20.575318,0.000,19.00000,33.000000,49.000000,100.000
duration_ms,89740.0,229144.365578,112945.780336,8586.000,173040.00000,213295.500000,264293.000000,5237295.000
danceability,89740.0,0.562166,0.176692,0.000,0.45000,0.576000,0.692000,0.985
energy,89740.0,0.634458,0.256606,0.000,0.45700,0.676000,0.853000,1.000
key,89740.0,5.283530,3.559912,0.000,2.00000,5.000000,8.000000,11.000
loudness,89740.0,-8.498994,5.221518,-49.531,-10.32225,-7.185000,-5.108000,4.532
mode,89740.0,0.636973,0.480875,0.000,0.00000,1.000000,1.000000,1.000
speechiness,89740.0,0.087442,0.113278,0.000,0.03600,0.048900,0.085900,0.965
acousticness,89740.0,0.328285,0.338321,0.000,0.01710,0.188000,0.625000,0.996
instrumentalness,89740.0,0.173415,0.323849,0.000,0.00000,0.000058,0.097625,1.000


In [35]:
print('Tempo igual a 0:', (df_catalogo['tempo'] == 0).sum())

print('Time signature igual a 0:', (df_catalogo['time_signature'] == 0).sum())

print('\nValores de time_signature:')
print(df_catalogo['time_signature'].value_counts().sort_index())

print('\nTempo y Time signature igual a 0:')
print(((df_catalogo['tempo'] == 0) & (df_catalogo['time_signature'] == 0)).sum())

print('\nLoudness positivos:', (df_catalogo['loudness'] > 0).sum())
print('Máximos de loudness:')
print(df_catalogo.nlargest(10, 'loudness')[['track_name', 'loudness']])

Tempo igual a 0: 157
Time signature igual a 0: 162

Valores de time_signature:
time_signature
0      162
1      846
3     7604
4    79543
5     1585
Name: count, dtype: int64

Tempo y Time signature igual a 0:
157

Loudness positivos: 68
Máximos de loudness:
                                              track_name  loudness
12070                                         Fate Dance     4.532
39156                                 Islamic Capitalism     3.156
25543                                          Time Bomb     2.574
39268                         A Better World Is Possible     1.864
24736                 TIPO HARIEL - SEI QUE E RARIDADE -     1.821
12388                                        Malfunction     1.795
39754                             Preaching for the Deaf     1.700
24606  AUTOMOTIVO DAS PIRANHA - VEM COMER MINHA TCHEC...     1.682
12283                                           The Mall     1.673
45552                       Mel Nattu Isai (From "Jodi")     1.416


In [36]:
df_catalogo = df_catalogo[df_catalogo['time_signature'] != 0].copy()

print('Filas restantes:', len(df_catalogo))
print('Registros eliminados:', 89740 - len(df_catalogo))
print('Time signatures iguales a 0:', (df_catalogo['time_signature'] == 0).sum())
print('Tempos iguales a 0:', (df_catalogo['tempo'] == 0).sum())
print('IDs únicos:', df_catalogo['track_id'].nunique())

Filas restantes: 89578
Registros eliminados: 162
Time signatures iguales a 0: 0
Tempos iguales a 0: 0
IDs únicos: 89578


In [37]:
print('IDs duplicados:', df_catalogo['track_id'].duplicated().sum())

print('Filas completamente duplicadas:', df_catalogo.duplicated().sum())

print('IDs únicos:', df_catalogo['track_id'].nunique())

print('Dimensiones:', df_catalogo.shape)

IDs duplicados: 0
Filas completamente duplicadas: 0
IDs únicos: 89578
Dimensiones: (89578, 20)


In [38]:
print('Géneros faltantes:', df_catalogo['track_genre'].isna().sum())
print('Géneros vacíos:', df_catalogo['track_genre'].str.strip().eq('').sum())

print('\nPopularidades faltantes:', df_catalogo['popularity'].isna().sum())
print('Popularidad mínima:', df_catalogo['popularity'].min())
print('Popularidad máxima:', df_catalogo['popularity'].max())
print('Tipo de popularidad:', df_catalogo['popularity'].dtype)

Géneros faltantes: 0
Géneros vacíos: 0

Popularidades faltantes: 0
Popularidad mínima: 0
Popularidad máxima: 100
Tipo de popularidad: int64


In [39]:
comparacion_popularidad = (
    df_catalogo[
        df_catalogo['track_id'].isin(ids_popularidad_variable)
    ][['track_id', 'popularity']]
    .merge(
        popularidad_consolidada,
        on = 'track_id',
        how = 'left',
        suffixes = ('_catalogo', '_calculada'),
        validate = 'one_to_one'
    )
)

print('IDs comparados:', len(comparacion_popularidad))

print(
    'Diferencias de popularidad:',
    (
        comparacion_popularidad['popularity_catalogo']
        != comparacion_popularidad['popularity_calculada']
    ).sum()
)

IDs comparados: 720
Diferencias de popularidad: 0


### **2.5 Revisión general y exportación**

In [40]:
print('Dimensiones:', df_catalogo.shape)
print('Filas duplicadas:', df_catalogo.duplicated().sum())
print('Valores faltantes:', df_catalogo.isna().sum().sum())
print('IDs únicos:', df_catalogo['track_id'].nunique())
print('IDs duplicados:', df_catalogo['track_id'].duplicated().sum())

print('\nFilas con tempo igual a 0:', (df_catalogo['tempo'] == 0).sum())
print('Distribución de time_signature:')
print(df_catalogo['time_signature'].value_counts().sort_index())

print('\nDistribución de popularity:')
print(df_catalogo['popularity'].describe())

Dimensiones: (89578, 20)
Filas duplicadas: 0
Valores faltantes: 0
IDs únicos: 89578
IDs duplicados: 0

Filas con tempo igual a 0: 0
Distribución de time_signature:
time_signature
1      846
3     7604
4    79543
5     1585
Name: count, dtype: int64

Distribución de popularity:
count    89578.000000
mean        33.197816
std         20.587911
min          0.000000
25%         19.000000
50%         33.000000
75%         49.000000
max        100.000000
Name: popularity, dtype: float64


In [41]:
ruta_exp = Path.cwd().parent / 'data'

df_catalogo.to_csv(ruta_exp / 'spotify-tracks-dataset-clean.csv', index = False, encoding = 'utf-8')

In [42]:
df_verificacion = pd.read_csv(ruta_exp / 'spotify-tracks-dataset-clean.csv')

print('Dimensiones:', df_verificacion.shape)
print('IDs únicos:', df_verificacion['track_id'].nunique())
print('Valores faltantes:', df_verificacion.isna().sum().sum())

Dimensiones: (89578, 20)
IDs únicos: 89578
Valores faltantes: 0
